#Importing Libraries

In [1]:
import pandas as pd
import numpy as np

# Loading and Reading the Data

In [2]:
df = pd.read_csv('/content/bank-additional-full.csv', sep=';')
print(df.head())

   age        job  marital    education  default housing loan    contact  \
0   56  housemaid  married     basic.4y       no      no   no  telephone   
1   57   services  married  high.school  unknown      no   no  telephone   
2   37   services  married  high.school       no     yes   no  telephone   
3   40     admin.  married     basic.6y       no      no   no  telephone   
4   56   services  married  high.school       no      no  yes  telephone   

  month day_of_week  ...  campaign  pdays  previous     poutcome emp.var.rate  \
0   may         mon  ...         1    999         0  nonexistent          1.1   
1   may         mon  ...         1    999         0  nonexistent          1.1   
2   may         mon  ...         1    999         0  nonexistent          1.1   
3   may         mon  ...         1    999         0  nonexistent          1.1   
4   may         mon  ...         1    999         0  nonexistent          1.1   

   cons.price.idx  cons.conf.idx  euribor3m  nr.employed

In [3]:
print(df.shape)
df.columns

(41188, 21)


Index(['age', 'job', 'marital', 'education', 'default', 'housing', 'loan',
       'contact', 'month', 'day_of_week', 'duration', 'campaign', 'pdays',
       'previous', 'poutcome', 'emp.var.rate', 'cons.price.idx',
       'cons.conf.idx', 'euribor3m', 'nr.employed', 'y'],
      dtype='object')

# Checking missing values

In [4]:
df.isnull().sum()

,0
age,0
job,0
marital,0
education,0
default,0
housing,0
loan,0
contact,0
month,0
day_of_week,0


# Checking duplicate rows

In [5]:
df.duplicated().sum()

np.int64(12)

**Remove Duplicates**

In [6]:
df = df.drop_duplicates()
df.duplicated().sum()

np.int64(0)

#Checking Unknown Values

In [7]:
(df == "unknown").sum()

,0
age,0
job,330
marital,80
education,1730
default,8596
housing,990
loan,990
contact,0
month,0
day_of_week,0


**Removing Unknown Values**

In [8]:
df_clean = df.copy()
df_clean = df_clean.replace("unknown", np.nan)

In [9]:
df_clean.isnull().sum()

,0
age,0
job,330
marital,80
education,1730
default,8596
housing,990
loan,990
contact,0
month,0
day_of_week,0


**Dropping Default column as it has too many null values**

In [10]:
df_final = df_clean.drop(columns=["default"])

df_final.isnull().sum()

,0
age,0
job,330
marital,80
education,1730
housing,990
loan,990
contact,0
month,0
day_of_week,0
duration,0


In [11]:
(df_clean == "unknown").sum()

,0
age,0
job,0
marital,0
education,0
default,0
housing,0
loan,0
contact,0
month,0
day_of_week,0


# Creating funnel KPIs

In [12]:
total_leads = len(df)
total_conversions = len(df[df['y'] == 'yes'])
conversion_rate = total_conversions / total_leads

print("Total Leads:", total_leads)
print("Total Conversions:", total_conversions)
print("Conversion Rate:", round(conversion_rate * 100, 2), "%")

Total Leads: 41176
Total Conversions: 4639
Conversion Rate: 11.27 %


# Channel-wise Conversion

In [13]:
channel_analysis = df_clean.groupby('contact')['y'].value_counts().unstack()

channel_analysis['conversion_rate'] = (
    channel_analysis['yes'] / (channel_analysis['yes'] + channel_analysis['no'])
)

channel_analysis

y,no,yes,conversion_rate
contact,,,
cellular,22283,3852,0.147389
telephone,14254,787,0.052324


# Job-wise Conversion

In [14]:
job_analysis = df_clean.groupby('job')['y'].value_counts().unstack()

job_analysis['conversion_rate'] = (
    job_analysis['yes'] / (job_analysis['yes'] + job_analysis['no'])
)

job_analysis.sort_values(by='conversion_rate', ascending=False)

y,no,yes,conversion_rate
job,,,
student,600,275,0.314286
retired,1284,434,0.252619
unemployed,870,144,0.142012
admin.,9068,1351,0.129667
management,2596,328,0.112175
technician,6009,730,0.108325
self-employed,1272,149,0.104856
housemaid,954,106,0.100000
entrepreneur,1332,124,0.085165


# Education-based conversion

In [15]:
edu_analysis = df_clean.groupby('education')['y'].value_counts().unstack()

edu_analysis['conversion_rate'] = (
    edu_analysis['yes'] / (edu_analysis['yes'] + edu_analysis['no'])
)

edu_analysis.sort_values(by='conversion_rate', ascending=False)

y,no,yes,conversion_rate
education,,,
illiterate,14,4,0.222222
university.degree,10495,1669,0.137208
professional.course,4645,595,0.113550
high.school,8481,1031,0.108389
basic.4y,3748,428,0.102490
basic.6y,2103,188,0.082060
basic.9y,5572,473,0.078246


# Funnel Drop-off Analysis

In [16]:
drop_off = df_clean['y'].value_counts()

drop_off_percentage = df_clean['y'].value_counts(normalize=True) * 100

print(drop_off)
print(drop_off_percentage)

y
no     36537
yes     4639
Name: count, dtype: int64
y
no     88.733728
yes    11.266272
Name: proportion, dtype: float64


# Exporting cleaned dataset

In [17]:
df_clean.to_csv("cleaned_marketing_data.csv", index=False)